# Penalty cap — optimal bid and success rate

A model of how a solver responds to the **penalty cap** $c_l$. A solver bids $s$ and wins the
auction iff $s \ge s_{\text{ref}}$, where the competing solvers' reference bid $s_{\text{ref}}$ is
drawn from a **configurable density on $[0, 2\mu]$** — uniform by default, or triangular, gaussian,
or rising (more mass at higher bids), selected with the *opponents* control. On a win the solver
realises a routing **surplus** $\sigma$ — a random draw of how good its execution turns out.
Conditional on winning:

- **reward** (paid on delivery): $R = \min(c_u,\, s - s_{\text{ref}})$
- **penalty** (paid on unset): $P = \min(c_l,\, s_{\text{ref}})$
- **deliver payoff** $R + \sigma - s$,  **unset payoff** $-P$.

**Two kinds of unset.**

- **Forced** — with probability $p_{\text{fail}}$ the routing fails outright (an infrastructure /
  submission failure) and the solver must unset and pay the penalty. This is the **base unset
  rate**: a fixed floor, independent of $\sigma$ and the caps.
- **Strategic** — among the non-forced draws the solver delivers only when the surplus clears a
  pre-planned threshold, otherwise it unsets rather than deliver at too large a loss:

  $$\text{deliver} \iff \sigma \ge \bar\sigma(s), \qquad
    \bar\sigma(s) = \mathbb{E}\big[\sigma^*(s, s_{\text{ref}}) \mid \text{win}\big], \qquad
    \sigma^* = s - R - P.$$

  This is the **routing-specific unset rate**, and it grows with the token's **volatility** — the
  spread $\sigma$ of the surplus distribution. A volatile token throws more low / negative draws
  below $\bar\sigma(s)$, so the solver unsets more. The threshold is fixed before $s_{\text{ref}}$
  is revealed, so it uses only the win-conditional *expectation* of the reference, never its
  realised value.

The surplus is drawn from a Gaussian $\mathcal{N}(\mu, \sigma)$ whose support includes negative
values, so a wide (volatile) distribution genuinely produces loss-making draws. Naming conventions:
 $c_l \leftrightarrow$ `penalty_cap` and $c_u \leftrightarrow$ `reward_cap_upper`.

**Units.** The model is **scale-invariant** — scaling every value ($s$, $s_{\text{ref}}$, $\sigma$,
$c_u$, $c_l$, $L$) by a common factor scales $s^*$ and $V$ by that factor and leaves the success
rate and the cap-to-surplus ratio $c_l^*/\mu$ unchanged. We therefore fix the **mean surplus
$\mu \equiv 1$** and read every quantity in units of $\mu$. The only volatility knob is then the
ratio $\sigma/\mu$ (the coefficient of variation): a genuinely more volatile token scales $\mu$ and
$\sigma$ together — a pure rescale that does nothing here — so the *shape* $\sigma/\mu$, not the
absolute level, is what drives the unset rate. Multiply the value axes by a representative $\mu$
(e.g. 50 bps) for an absolute reading.

**Figure.** For each penalty cap $c_l$ the solver re-optimises its bid
$s^*(c_l) = \arg\max_s \Pi(s)$. We sweep $c_l$ along the x-axis and track:

- **top-left panel:** the optimal bid $s^*(c_l)$,
- **top-right panel:** the **success rate** $P(\text{deliver}\mid\text{win}) = 1 - \text{unset rate}$,
  whose shortfall from $1$ splits into the base rate $p_{\text{fail}}$ and the routing-specific
  (volatility-driven) rate,
- **bottom panel:** the **expected value** of the won auction (in units of $\mu$), which trades
  execution *quality* off against execution *reliability* (see below).

**Value.** The bid $s$ is itself the surplus delivered to the user on a successful settlement, so
the expected value per won auction is

$$V(c_l) \;=\; \underbrace{P(\text{deliver}\mid\text{win})\cdot s^*(c_l)}_{\text{expected delivered surplus}}
  \;-\; \underbrace{\big(1 - P(\text{deliver}\mid\text{win})\big)\cdot L}_{\text{expected failure cost}},$$

where $L$ is the (fixed) **cost of one unset** — what the user loses to delay / re-quote when the
trade fails to settle (the vol-drag anchor is $L \sim \sigma$, i.e. $L/\mu \sim \sigma/\mu$). Raising
$c_l$ lowers $s^*$ (worse price, lower execution quality) but raises the success rate (better
reliability); $V$ balances the two and peaks at the optimal cap $c_l^*$, marked on all three panels.

Controls set the **relative volatility** $\sigma/\mu$, the base unset rate $p_{\text{fail}}$, the
reward cap $c_u/\mu$, the unset cost $L/\mu$, and the competing-solver bid shape (*opponents*); the
$c_l$ sweep range auto-scales with $\mu$.

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

In [ ]:
# ----------------------------------------------------------------------
# Model.  sigma is the routing SURPLUS and may be negative (a poor draw
# means delivering at a loss).  With probability p_fail the routing is
# forced to unset; otherwise the solver delivers iff the surplus clears
# its pre-planned threshold sigma_bar(s) and unsets below it.
# ----------------------------------------------------------------------
def gaussian_surplus(mu, sigstd, n, k=4.0):
    """Discretised Gaussian surplus on [mu - k*sigstd, mu + k*sigstd].
    The support spans negative surplus, so a wide draw can come out a loss."""
    edges = np.linspace(mu - k * sigstd, mu + k * sigstd, n + 1)
    mids = 0.5 * (edges[:-1] + edges[1:])
    w = np.exp(-0.5 * ((mids - mu) / sigstd) ** 2)
    return mids, w / w.sum()


def reference_weights(shape, S_ref):
    """Competing-solver bid density evaluated on the S_ref grid (atoms on
    [0, max]).  Returns weights that sum to 1.

    - 'uniform'   flat — every reference bid equally likely;
    - 'triangle'  peaked at the midpoint, zero at the ends;
    - 'gaussian'  bell at the midpoint (std = 20% of the range);
    - 'rising'    linearly more mass at higher reference bids.

    The support is always [0, max]; only the *shape* of the density changes,
    so changing it is a clean apples-to-apples test of the opponent model."""
    x = S_ref
    lo, hi = x[0], x[-1]
    mid = 0.5 * (lo + hi)
    if shape == "uniform":
        w = np.ones_like(x)
    elif shape == "triangle":
        w = np.maximum(0.0, 1.0 - np.abs(x - mid) / (0.5 * (hi - lo)))
    elif shape == "gaussian":
        w = np.exp(-0.5 * ((x - mid) / (0.2 * (hi - lo))) ** 2)
    elif shape == "rising":
        w = (x - lo) + 1e-9
    else:
        raise ValueError(f"unknown reference shape {shape!r}")
    return w / w.sum()


def _tail_moments(sig, p):
    """Return tail(tau) -> (P(sigma >= tau), E[sigma * 1{sigma >= tau}]),
    vectorised over an array of thresholds tau, via suffix sums over the
    sorted surplus atoms."""
    order = np.argsort(sig)
    s = sig[order]; pp = p[order]
    suf_p = np.concatenate([np.cumsum(pp[::-1])[::-1], [0.0]])
    suf_s = np.concatenate([np.cumsum((pp * s)[::-1])[::-1], [0.0]])

    def tail(tau):
        idx = np.searchsorted(s, tau, side="left")  # count of atoms < tau
        return suf_p[idx], suf_s[idx]

    return tail


def model_curves(routing, p_fail, S, S_ref, w_ref, c_u, c_l):
    """Expected payoff Pi(s) and success rate (conditional on winning) over
    the bid grid S.

    Pi(s) = sum_{s_ref} w_ref * 1{win} * E_sigma[payoff]   (losing -> 0),
    where w_ref is the competing-solver bid density (sums to 1).  With prob
    p_fail the routing is forced to unset (-P).  Otherwise the solver
    delivers iff sigma >= sigma_bar(s), so the non-forced payoff is
    q R + surplus - q s - (1 - q) P, with q = P(sigma >= sigma_bar(s)).

    Note: w_ref should be at least as fine as the bid grid S, otherwise bids
    falling between reference atoms sit on a flat ridge of Pi and the argmax
    flips between near-ties as c_l moves -> a jagged s*(c_l) artifact."""
    sig, p = routing
    tail = _tail_moments(sig, p)

    Sm = S[:, None]; Sr = S_ref[None, :]; W = w_ref[None, :]
    win = Sm >= Sr                                  # (ns, nr)
    R = np.minimum(c_u, Sm - Sr)                    # reward where win
    P = np.minimum(c_l, Sr)                         # penalty by s_ref
    winw = win * W                                  # win mass by (s, s_ref)
    wmass = winw.sum(1)                             # P(win) under the density

    # sigma_bar(s) = E[sigma*(s, s_ref) | win], one threshold per bid s.
    sbar = (winw * (Sm - R - P)).sum(1) / np.maximum(wmass, 1e-12)

    q, surplus = tail(sbar)                         # deliver prob & delivered surplus (non-forced)
    g_ok = q[:, None] * R + surplus[:, None] - q[:, None] * Sm - (1 - q)[:, None] * P
    g = (1 - p_fail) * g_ok + p_fail * (-P)         # blend non-forced draws with forced unset
    Pi = (np.where(win, g, 0.0) * W).sum(1)         # density-weighted over s_ref
    success = (1 - p_fail) * q                      # P(deliver | win)
    return dict(Pi=Pi, success=success)


def optima_for_cap(routing, p_fail, S, S_ref, w_ref, c_u, c_l):
    """Optimal bid s* (argmax expected payoff) and the success rate at s*."""
    cur = model_curves(routing, p_fail, S, S_ref, w_ref, c_u, c_l)
    i = int(np.argmax(cur["Pi"]))
    return float(S[i]), float(cur["success"][i])


def cap_sweep(routing, p_fail, S, S_ref, w_ref, c_u, c_l_grid):
    """Optimal bid and success rate for each penalty cap in the grid."""
    out = [optima_for_cap(routing, p_fail, S, S_ref, w_ref, c_u, c_l) for c_l in c_l_grid]
    s_star, success = (np.array(col) for col in zip(*out))
    return dict(s_star=s_star, success=success)

In [ ]:
# ----------------------------------------------------------------------
# Interactive figure: optimal bid s*(c_l), success rate at s*, and the
# expected value V(c_l) as the penalty cap c_l sweeps the x-axis.
#
# Units.  The model is scale-invariant: scaling s, s_ref, sigma, c_u, c_l, L
# by k scales s* and V by k and leaves success and c_l*/mu unchanged.  So the
# absolute scale is cosmetic and we normalise the mean surplus to mu = 1.
# Everything is then in units of mu: the vol slider is sigma/mu (coefficient
# of variation -- the only honest "volatility" knob, since a genuinely more
# volatile token scales mu and sigma together, a pure rescale invisible here),
# and c_u, L, the rival-bid range are multiples of mu.  Set MU below to a
# representative figure (e.g. 50 bps) to read the value axes in absolute
# terms; it is a pure rescale and changes none of the dynamics.
#
# Value.  The bid s is itself the surplus delivered to the user on a
# successful settlement, so the expected value per won auction is
#     V(c_l) = success * s*            (expected delivered surplus)
#            - L * (1 - success)       (expected failure cost)
# where L is the cost of one unset (delay / re-quote).  s* falls as the cap
# rises (worse price = lower execution quality) while success rises (better
# reliability); V trades the two off and peaks at c_l*.  The vol-drag anchor
# for the unset cost is L ~ sigma, i.e. L/mu ~ sigma/mu; crank L up (the
# liquidation regime) and c_l* saturates where success plateaus, not at the edge.
#
# Sliders: relative vol sigma/mu, base unset rate p_fail, reward cap c_u/mu,
# unset cost L/mu, and the competing-solver bid shape.
# ----------------------------------------------------------------------
MU        = 1.0                            # mean surplus = the value unit (set to e.g. 50 for bps)
S_REF_MAX = 2.0 * MU                       # competing solvers bid s_ref in [0, 2*mu] (mean rival ~ mu)
S      = np.linspace(0.0, S_REF_MAX, 201)  # the solver's own candidate bids s
# Keep the s_ref grid finer than the bid grid S: a coarser reference strands
# bids between atoms on a flat ridge of Pi, jittering argmax (s*) as c_l moves.
N_REF  = 401
S_ref  = np.linspace(0.0, S_REF_MAX, N_REF)
N_CAPS = 201                               # resolution of the c_l sweep
N_BINS = 200                               # surplus discretisation bins

plt.ioff()
fig, axd = plt.subplot_mosaic([["bid", "success"],
                               ["value", "value"]],
                              figsize=(10.5, 7.5), constrained_layout=True)
plt.ion()
axL, axR, axV = axd["bid"], axd["success"], axd["value"]
fig.canvas.header_visible = False

(lineL,) = axL.plot([], [], color="dodgerblue", lw=2, label="optimal bid $s^*$")
mu_line  = axL.axhline(0.0, ls="--", lw=1, color="seagreen",   alpha=0.8)
emu_line = axL.axhline(0.0, ls="--", lw=1, color="darkorange", alpha=0.8)
optL_vline = axL.axvline(0.0, ls=":", lw=1.2, color="black", alpha=0.7)
(optL_marker,) = axL.plot([], [], "o", color="black", ms=6, zorder=5)
(lineR,) = axR.plot([], [], color="dodgerblue", lw=2, label="success rate")
ceil_line = axR.axhline(0.0, ls="--", lw=1, color="gray", alpha=0.8)
optR_vline = axR.axvline(0.0, ls=":", lw=1.2, color="black", alpha=0.7)
(optR_marker,) = axR.plot([], [], "o", color="black", ms=6, zorder=5)

(lineV_deliv,) = axV.plot([], [], color="seagreen", lw=2,
                          label="delivered surplus  $\\mathrm{success}\\cdot s^*$")
(lineV_cost,)  = axV.plot([], [], color="firebrick", lw=2,
                          label="$-$failure cost  $-L(1-\\mathrm{success})$")
(lineV_net,)   = axV.plot([], [], color="dodgerblue", lw=2.5, label="net value  $V$")
(opt_marker,)  = axV.plot([], [], "o", color="black", ms=7, zorder=5)
opt_vline = axV.axvline(0.0, ls=":", lw=1.2, color="black", alpha=0.7)
axV.axhline(0.0, ls="-", lw=0.8, color="gray", alpha=0.5)

axL.set(xlabel="penalty cap  $c_l$  $[\\mu]$", ylabel="optimal bid  $s^*$  $[\\mu]$", title="Optimal bid")
axR.set(xlabel="penalty cap  $c_l$  $[\\mu]$", ylabel="success rate   $P(\\mathrm{deliver}\\mid\\mathrm{win})$",
        title="Success rate at $s^*$", ylim=(0, 1.02))
axV.set(xlabel="penalty cap  $c_l$  $[\\mu]$", ylabel="expected value per won auction  $[\\mu]$",
        title="Net value at $s^*$")

sl = dict(
    cov    = widgets.FloatSlider(value=0.5, min=0.01, max=1.0, step=0.01, description="σ/μ (rel. vol)",
                                 readout_format=".2f"),
    p_fail = widgets.FloatSlider(value=0.05,min=0,    max=0.5, step=0.01, description="p_fail (base)",
                                 readout_format=".2f"),
    c_u    = widgets.FloatSlider(value=1.5, min=0,    max=4.0, step=0.05, description="c_u (×μ)",
                                 readout_format=".2f"),
    L      = widgets.FloatSlider(value=0.5, min=0,    max=8.0, step=0.1,  description="L (×μ)",
                                 readout_format=".2f"),
    shape  = widgets.Dropdown(options=["uniform", "triangle", "gaussian", "rising"],
                              value="uniform", description="opponents"),
)

def update(_=None):
    v = {k: w.value for k, w in sl.items()}
    mu     = MU
    sigstd = v["cov"] * MU
    c_u    = v["c_u"] * MU
    L      = v["L"]   * MU
    routing = gaussian_surplus(mu, sigstd, N_BINS)
    w_ref = reference_weights(v["shape"], S_ref)
    c_l_max = max(0.2 * MU, 1.1 * (1 - v["p_fail"]) * mu)
    c_l_grid = np.linspace(0, c_l_max, N_CAPS)
    sw = cap_sweep(routing, v["p_fail"], S, S_ref, w_ref, c_u, c_l_grid)

    lineL.set_data(c_l_grid, sw["s_star"])
    lineR.set_data(c_l_grid, sw["success"])

    # value: expected delivered surplus minus expected failure cost
    deliv    = sw["success"] * sw["s_star"]
    failcost = L * (1 - sw["success"])
    Vnet     = deliv - failcost
    i_opt    = int(np.argmax(Vnet))
    c_opt, V_opt = float(c_l_grid[i_opt]), float(Vnet[i_opt])
    lineV_deliv.set_data(c_l_grid, deliv)
    lineV_cost.set_data(c_l_grid, -failcost)
    lineV_net.set_data(c_l_grid, Vnet)
    opt_marker.set_data([c_opt], [V_opt])

    # mark the optimal cap c_l* on every panel
    for vline in (optL_vline, optR_vline, opt_vline):
        vline.set_xdata([c_opt, c_opt])
        vline.set_label(f"optimal cap  $c_l^*$ = {c_opt:.2f}")
    optL_marker.set_data([c_opt], [sw["s_star"][i_opt]])
    optR_marker.set_data([c_opt], [sw["success"][i_opt]])

    top = 1 - v["p_fail"]
    ceil_line.set_ydata([top, top])
    ceil_line.set_label(f"no strategic unset  $1-p_{{fail}}$ = {top:.2f}")

    emu = (1 - v["p_fail"]) * mu                        # E[surplus] vs E[surplus | success]=mu
    mu_line.set_ydata([mu, mu]); mu_line.set_label(f"$\\mu$ = {mu:.2f}")
    emu_line.set_ydata([emu, emu]); emu_line.set_label(f"$(1-p_{{fail}})\\mu$ = {emu:.2f}")

    axL.set_xlim(0, c_l_max); axR.set_xlim(0, c_l_max); axV.set_xlim(0, c_l_max)
    axL.set_ylim(0, max(mu, float(sw["s_star"].max())) * 1.08)
    vlo = min(0.0, float((-failcost).min()), float(Vnet.min()))
    vhi = max(0.0, float(deliv.max()), float(Vnet.max()))
    pad = 0.08 * (vhi - vlo if vhi > vlo else 1.0)
    axV.set_ylim(vlo - pad, vhi + pad)
    axL.legend(loc="best", fontsize=8)
    axR.legend(loc="best", fontsize=8)
    axV.legend(loc="best", fontsize=8)
    fig.suptitle(f"σ/μ={v['cov']:.2f}   p_fail(base)={v['p_fail']:.2f}   "
                 f"c_u/μ={v['c_u']:.2f}   L/μ={v['L']:.2f}   "
                 f"opponents={v['shape']}   (μ≡{MU:.0f})", fontsize=9)
    fig.canvas.draw_idle()

for w in sl.values():
    w.observe(update, names="value")
update()

display(widgets.HBox([widgets.VBox(list(sl.values())), fig.canvas]))

### Reading the figure

The top-right panel decomposes the unset rate top-down:

- the gap from **1.0** to the dashed line **$1 - p_{\text{fail}}$** is the **base** (forced) unset
  rate — the floor the solver can't avoid;
- the gap from the dashed line down to the **success curve** is the **routing-specific** unset rate
  — the strategic unsets driven by volatility.

- **Success rate.** A cheap penalty makes the threshold $\bar\sigma(s)$ high, so the solver unsets
  on more low / negative draws and the success rate sits well below $1 - p_{\text{fail}}$. As $c_l$
  rises the threshold falls and the success rate climbs. It does **not** reach $1 - p_{\text{fail}}$:
  even at a large cap the solver still unsets on negative-surplus draws, because the penalty is
  capped at $s_{\text{ref}}$, so paying out of pocket on a negative draw is never worth it.
- **Relative volatility ($\sigma/\mu$).** Raising it widens the surplus distribution relative to its
  mean, pushes more mass below $\bar\sigma(s)$, and shifts the whole success curve **down** — a
  higher-CoV token unsets more, at every cap.
- **Optimal bid.** A cheaper unset option (small $c_l$) lets the solver bid a little more
  aggressively to win; the bid settles down as the cap rises and the option loses value.
- **Reference lines (top-left).** The dashed lines mark the expected routing surplus,
  $(1-p_{\text{fail}})\mu$, and its value conditional on the routing not failing, $\mu$; comparing
  $s^*$ against them shows how the bid tracks the surplus the solver expects to capture.

- **Net value (bottom).** The green curve is the **expected delivered surplus** $\,\text{success}\cdot s^*$
  — it falls once the success gain no longer offsets the dropping bid. The red curve is the
  **negative failure cost** $-L(1-\text{success})$ — it rises (toward zero) as the cap buys
  reliability. Their sum, the blue **net value** $V$, peaks at the **optimal cap $c_l^*$**
  (black marker / dotted line). Pushing $L$ up makes failures more expensive and shifts $c_l^*$
  to a **higher** cap, saturating where the success rate plateaus; pushing $L$ to $0$ collapses $V$
  onto the delivered-surplus curve and the optimum sits wherever $\text{success}\cdot s^*$ is
  largest. Raising $\sigma/\mu$ lowers the success rate but leaves $L$ untouched (it has its own
  slider); the default pins $L/\mu = \sigma/\mu$ as the vol-drag anchor, but the two do not
  auto-couple, so a deliberately riskier token must also be made costlier to leave unsettled by hand.